# PII Redaction with YAKE + NER-Filtered Keywords + Regex on SageMaker

This notebook is a variant of `redaction_yake_sagemaker.ipynb` with one key difference:
**not all YAKE-extracted keywords are redacted — only those that a NER model classifies as `PER`, `LOC`, `ORG`, or `MISC`**.

**Approach:**
1. **Regex pre-pass** — catches structured PII (SSN, dates, phone numbers, postal codes, IDs)
2. **YAKE keyword extraction** — surfaces candidate terms (names, organisations, domain terms)
3. **NER filtering** — runs `ner_dl` (CoNLL 2003, 4-class) on the same text; keeps only YAKE keywords
   that overlap with `PER`, `LOC`, `ORG`, or `MISC` NER entities
4. **Redaction** — replaces regex PII + NER-filtered YAKE keywords with `[REDACTED]`

**Why filter YAKE through NER?**
- YAKE is unsupervised and surfaces *all* statistically prominent terms (e.g. `chest pain`, `complaint`, `contract clause`) — most of which are not PII
- NER narrows the set to named entities that are privacy-sensitive
- The combination gives broader recall than NER alone (YAKE catches unusual names/terms) while avoiding over-redaction of non-PII keywords

**SageMaker v2.x compatible** — uses `sagemaker>=2.0` SDK APIs.

## 1. Environment Setup

In [1]:
import subprocess, sys, os

java_check = subprocess.run(["java", "-version"], capture_output=True)
if java_check.returncode != 0:
    print("Java not found — installing default-jdk ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jdk"], check=True)
    print("Java installed.")
else:
    print("Java already available:", java_check.stderr.decode().split('\n')[0])

java_home_candidates = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-amazon-corretto",
    "/usr/lib/jvm/java-8-openjdk-amd64",
    "/usr/lib/jvm/java-8-amazon-corretto",
]
for candidate in java_home_candidates:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        break
else:
    result = subprocess.run(
        ["java", "-XshowSettings:all", "-version"],
        capture_output=True, text=True
    )
    for line in result.stderr.splitlines():
        if "java.home" in line:
            java_home = line.split("=")[1].strip()
            if java_home.endswith("/jre"):
                java_home = java_home[:-4]
            os.environ["JAVA_HOME"] = java_home
            break

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET"))

Java already available: openjdk version "11.0.30" 2026-01-20
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [2]:
!pip install -q \
    "sagemaker>=2.0,<3.0" \
    "pyspark>=3.3,<3.6" \
    "spark-nlp" \
    "spark-nlp-display" \
    "boto3"

print("All packages installed.")

All packages installed.


## 2. SageMaker v2.x Session Setup

In [3]:
import boto3
import sagemaker
from sagemaker import get_execution_role

boto_session = boto3.Session()
sm_session   = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sagemaker_session=sm_session)
except Exception:
    role = os.environ.get("SAGEMAKER_ROLE", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")
    print("WARNING: Could not auto-detect execution role. Set SAGEMAKER_ROLE env var.")

region = boto_session.region_name or sm_session.boto_region_name
bucket = sm_session.default_bucket()
prefix = "spark-nlp/redaction-yake-ner"

print(f"SageMaker SDK version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Default S3 bucket     : {bucket}")
print(f"Execution role ARN    : {role}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/cjen/.config/sagemaker/config.yaml
SageMaker SDK version : 2.257.1
Region                : us-west-2
Default S3 bucket     : sagemaker-us-west-2-493644444178
Execution role ARN    : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd


## 3. Import Spark NLP & Start Spark Session

> **Note:** `pyspark.sql.functions` contains a built-in `bucket()` function.
> Always use `import pyspark.sql.functions as F` (alias) rather than `from pyspark.sql.functions import *`
> to avoid silently overwriting the `bucket` string variable assigned in the cell above.

In [4]:
import sys
print("Python executable:", sys.executable)

import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline

import pyspark.sql.functions as F
from pyspark.sql.types import StringType, ArrayType

import pandas as pd
import re
import warnings

warnings.filterwarnings("ignore")
print("Spark NLP version:", sparknlp.version())

Python executable: /home/cjen/mySageMaker/ML/spark-nlp/redaction/key/.venv/bin/python
Spark NLP version: 6.3.3


In [5]:
params = {
    "spark.driver.memory"           : "16G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize"    : "2000M",
    "spark.ui.enabled"              : "false",
}

spark = sparknlp.start(params=params)
spark

26/03/22 14:23:47 WARN Utils: Your hostname, MindyJen1008 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/22 14:23:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/cjen/mySageMaker/ML/spark-nlp/redaction/key/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/cjen/.ivy2/cache
The jars for the packages stored in: /home/cjen/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a4804a7f-2073-48ed-851f-472ba0145b7c;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	f

## 4. Sample Data

Same texts as `redaction_sagemaker.ipynb` — they contain names, dates, SSNs, addresses, and IDs.

In [6]:
#sample_texts = [
#    "Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.",
#    "Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple Street, Toronto.",
#    "The contract signed by Alice Wong and Bob Martin on January 15 2023 includes a clause for New York jurisdiction.",
#    "Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 on 09/22/2024 at 3:45 PM.",
#    "Invoice #INV-2024-88 issued to Acme Corp, attention David Brown, 100 King Street West, Vancouver, BC V6B 1A1.",
#]

sample_texts = [
    "Quick, listened well, kind, helped me feel better, took care of me quickly",
    "The nurse who did the triage was very rude to my friend. Already she is stressed for being in an embulatory environment and in an other country (she is from Latino America). She doesn't understand French or English and she felt stress for coming here from the start. When the nurse took her vitals she hurt her arm. Now my friend has marks on her arm and she felt really bad after.",
    "The care that they give is excellent",
    "Triage rapide",
    "The nurses were very caring and supportive",
    "Staff was very caring and understanding. Doctor was very patient and calm. Same for the RN and PAB",
    "I was treated in a reasonable amount of time",
    "Everything is ccept the waiting time",
    "All medical personnel and staff were extra attentive, professional,responsive and pro-active in letting me know ahead of time what test were being done and would be done including time frame. Exceptional processes and patient satisfaction in mind"
]

pdf      = pd.DataFrame(sample_texts, columns=["text"])
spark_df = spark.createDataFrame(pdf)
spark_df.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|text                                                                                                                                                                                                                                                                                                                                                                                         |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 5. Regex Pre-pass — Structured PII

YAKE is a statistical keyword extractor — it will surface names and organisations but is unlikely to flag
numeric patterns such as SSNs, ISO dates, or phone numbers because these have low term-frequency scores.

The regex pre-pass catches:
| Pattern | Example |
|---|---|
| SSN | `123-45-6789` |
| ISO date | `2024-11-01` |
| US/CA date | `09/22/2024` |
| Written date | `March 4 1975`, `January 15 2023` |
| Time | `3:45 PM` |
| Postal code | `V6B 1A1` |
| Phone | `(416) 555-0123` |
| Employee/complaint IDs | `E-4892`, `CR-00312`, `INV-2024-88` |

In [7]:
# Ordered from most-specific to least-specific to avoid partial overlaps
REGEX_PII_PATTERNS = [
    # SSN  (must come before plain numbers)
    (r'\b\d{3}-\d{2}-\d{4}\b',                          '[SSN]'),
    # ISO date  2024-11-01
    (r'\b\d{4}-\d{2}-\d{2}\b',                          '[DATE]'),
    # US/CA date  09/22/2024  or  9/1/2024
    (r'\b\d{1,2}/\d{1,2}/\d{4}\b',                      '[DATE]'),
    # Written date with 4-digit year  March 4 1975 / January 15 2023
    (r'\b(?:January|February|March|April|May|June|July|August|September|'
     r'October|November|December)\s+\d{1,2}\s+\d{4}\b', '[DATE]'),
    # Time  3:45 PM / 14:30
    (r'\b\d{1,2}:\d{2}(?:\s?[AP]M)?\b',                 '[TIME]'),
    # Canadian postal code  V6B 1A1
    (r'\b[A-Z]\d[A-Z]\s?\d[A-Z]\d\b',                  '[POSTAL_CODE]'),
    # Phone  (416) 555-0123 / 416-555-0123 / +1-800-555-0123
    (r'(?:\+?1[-. ])?(?:\(?\d{3}\)?[-. ])\d{3}[-. ]\d{4}\b', '[PHONE]'),
    # Alphanumeric IDs: E-4892, CR-00312, INV-2024-88
    (r'\b[A-Z]{1,4}-[A-Z0-9]{2,}-?[0-9]{0,4}\b',       '[ID]'),
]

def regex_redact(text):
    """Apply all regex PII patterns left-to-right."""
    if not text:
        return text
    for pattern, replacement in REGEX_PII_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text

regex_redact_udf = F.udf(regex_redact, StringType())

# Apply regex redaction first
regex_df = spark_df.withColumn("regex_redacted", regex_redact_udf(F.col("text")))
regex_df.select("text", "regex_redacted").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|text                                                                                                                                                                                                                                     

## 6. Combined YAKE + NER Pipeline

We build a single Spark NLP `Pipeline` that runs both annotators on the same tokenised input:

| Stage | Output column | Purpose |
|---|---|---|
| `DocumentAssembler` | `document` | Wrap raw text |
| `SentenceDetector` | `sentence` | Segment into sentences |
| `Tokenizer` | `token` | Split into tokens |
| `YakeKeywordExtraction` | `keywords` | Unsupervised keyword candidates |
| `WordEmbeddingsModel` (`glove_100d`) | `embeddings` | Word vectors for NER |
| `NerDLModel` (`ner_dl`) | `ner` | Token-level NER tags (B-PER, I-LOC, …) |
| `NerConverter` | `ner_chunk` | Merge IOB tags → named-entity chunks |

The `ner_dl` model is trained on CoNLL 2003 and produces four entity types:
`PER` (person), `LOC` (location), `ORG` (organisation), `MISC` (miscellaneous named entity).

Only YAKE keywords that **overlap** with one of those four NER entity types will be redacted.

In [8]:
# ── Stage 1: Raw text → Document ─────────────────────────────────────────────
document_assembler = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

# ── Stage 2: Sentence segmentation ───────────────────────────────────────────
sentence_detector = (
    SentenceDetector()
    .setInputCols(["document"])
    .setOutputCol("sentence")
)

# ── Stage 3: Tokenisation ─────────────────────────────────────────────────────
tokenizer = (
    Tokenizer()
    .setInputCols(["sentence"])
    .setOutputCol("token")
)

# ── Stage 4: YAKE keyword extraction ─────────────────────────────────────────
# setThreshold: keywords with score BELOW threshold are kept.
# Lower threshold → stricter, fewer keywords (names/orgs rank well).
yake = (
    YakeKeywordExtraction()
    .setInputCols(["token"])
    .setOutputCol("keywords")
    .setMinNGrams(1)
    .setMaxNGrams(3)
    .setNKeywords(30)
    .setThreshold(1.5)
    .setWindowSize(3)
)

# ── Stage 5: Word embeddings (required by NerDL) ─────────────────────────────
embeddings = (
    WordEmbeddingsModel.pretrained("glove_100d")
    .setInputCols(["sentence", "token"])
    .setOutputCol("embeddings")
)

# ── Stage 6: NER model (CoNLL 2003 — PER / LOC / ORG / MISC) ─────────────────
ner = (
    NerDLModel.pretrained("ner_dl")
    .setInputCols(["sentence", "token", "embeddings"])
    .setOutputCol("ner")
)

# ── Stage 7: Convert IOB NER tags → entity chunks ─────────────────────────────
ner_converter = (
    NerConverter()
    .setInputCols(["sentence", "token", "ner"])
    .setOutputCol("ner_chunk")
)

combined_pipeline = Pipeline(
    stages=[
        document_assembler,
        sentence_detector,
        tokenizer,
        yake,
        embeddings,
        ner,
        ner_converter,
    ]
)

print("Combined YAKE + NER pipeline assembled.")

glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[ | ]

26/03/22 14:33:05 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/22 14:33:05 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


glove_100d download started this may take some time.
Approximate size to download 145.3 MB
Download done! Loading the resource.
[OK!]
ner_dl download started this may take some time.
Approximate size to download 13.6 MB
[ | ]

26/03/22 14:33:10 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/22 14:33:10 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/22 14:33:11 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_dl download started this may take some time.
Approximate size to download 13.6 MB
Download done! Loading the resource.
[ / ]

2026-03-22 14:33:15.082917: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-22 14:33:15.210898: W external/org_tensorflow/tensorflow/core/common_runtime/colocation_graph.cc:1218] Failed to place the graph without changing the devices of some resources. Some of the operations (that had to be colocated with resource generating operations) are not supported on the resources' devices. Current candidate devices are [
  /job:localhost/replica:0/task:0/device:CPU:0].
See below for details of this colocation group:
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=-1 requested_device_name_='/device:GPU:0' assigned_device_nam

[OK!]
Combined YAKE + NER pipeline assembled.


In [9]:
empty_df       = spark.createDataFrame([[""]], ["text"])
combined_model = combined_pipeline.fit(empty_df)
print("Model ready.")

Model ready.


## 7. Run Inference & Inspect YAKE Keywords and NER Entities

In [10]:
result_df = combined_model.transform(spark_df)

# Deduplicate YAKE keywords per row
result_df = result_df.withColumn(
    "unique_keywords", F.array_distinct(F.col("keywords.result"))
)

# ── Inspect YAKE keywords ─────────────────────────────────────────────────────
print("=== YAKE Keywords ===")
keywords_flat = result_df.select(
    F.col("text"),
    F.explode("keywords").alias("kw")
).select(
    "text",
    F.col("kw.result").alias("keyword"),
    F.col("kw.metadata")["score"].cast("double").alias("score"),
).orderBy("text", "score")
keywords_flat.show(60, truncate=80)

# ── Inspect NER entities ──────────────────────────────────────────────────────
print("=== NER Entities (PER / LOC / ORG / MISC) ===")
ner_flat = result_df.select(
    F.col("text"),
    F.explode("ner_chunk").alias("chunk")
).select(
    "text",
    F.col("chunk.result").alias("entity_text"),
    F.col("chunk.metadata")["entity"].alias("entity_label"),
).orderBy("text", "entity_label")
ner_flat.show(60, truncate=80)

=== YAKE Keywords ===


+--------------------------------------------------------------------------------+---------------------+-------------------+
|                                                                            text|              keyword|              score|
+--------------------------------------------------------------------------------+---------------------+-------------------+
|All medical personnel and staff were extra attentive, professional,responsive...|                 time|0.29615709559935455|
|All medical personnel and staff were extra attentive, professional,responsive...|                 done|0.29615709559935455|
|All medical personnel and staff were extra attentive, professional,responsive...|                 done|0.29615709559935455|
|All medical personnel and staff were extra attentive, professional,responsive...|                 time|0.29615709559935455|
|All medical personnel and staff were extra attentive, professional,responsive...|                 mind| 0.5689633223058038|


+--------------------------------------------------------------------------------+--------------+------------+
|                                                                            text|   entity_text|entity_label|
+--------------------------------------------------------------------------------+--------------+------------+
|Staff was very caring and understanding. Doctor was very patient and calm. Sa...|           PAB|         ORG|
|The nurse who did the triage was very rude to my friend. Already she is stres...|Latino America|         LOC|
|The nurse who did the triage was very rude to my friend. Already she is stres...|        French|        MISC|
|The nurse who did the triage was very rude to my friend. Already she is stres...|       English|        MISC|
+--------------------------------------------------------------------------------+--------------+------------+



## 8. Filter YAKE Keywords by NER Labels

A YAKE keyword is **kept for redaction** if it overlaps (case-insensitive, token-level) with at least one
NER entity labelled `PER`, `LOC`, `ORG`, or `MISC`.

Overlap is checked as:
- exact match: `yake_kw == ner_entity`
- YAKE keyword is a sub-phrase of a NER entity: `yake_kw in ner_entity`
- NER entity is a sub-phrase of a YAKE keyword: `ner_entity in yake_kw`

This ensures multi-word names like *'John Smith'* are caught even if YAKE extracts *'John'* and NER
returns *'John Smith'* (or vice versa).

Keywords that do **not** match any PER/LOC/ORG/MISC entity (e.g. `chest pain`, `contract clause`,
`invoice`) are **not redacted**.

In [11]:
NER_PII_LABELS = {"PER", "LOC", "ORG", "MISC"}

@F.udf(ArrayType(StringType()))
def extract_ner_pii_entities(ner_chunks):
    """Return the text of all NER chunks whose entity label is PER/LOC/ORG/MISC."""
    if not ner_chunks:
        return []
    return [
        chunk.result
        for chunk in ner_chunks
        if chunk.metadata.get("entity", "") in NER_PII_LABELS
    ]


@F.udf(ArrayType(StringType()))
def filter_yake_by_ner(yake_keywords, ner_pii_entities):
    """
    Keep only YAKE keywords that overlap with NER-detected PER/LOC/ORG/MISC entities.

    A YAKE keyword is kept when (case-insensitive):
      - it exactly matches a NER entity, OR
      - it is a sub-phrase of a NER entity, OR
      - a NER entity is a sub-phrase of it
    """
    if not yake_keywords or not ner_pii_entities:
        return []
    ner_lower = [e.lower() for e in ner_pii_entities]
    filtered = []
    for kw in yake_keywords:
        kw_lower = kw.lower()
        if any(
            kw_lower == ne or kw_lower in ne or ne in kw_lower
            for ne in ner_lower
        ):
            filtered.append(kw)
    return list(dict.fromkeys(filtered))  # deduplicate, preserve insertion order


# ── Apply both UDFs ───────────────────────────────────────────────────────────
result_df = (
    result_df
    .withColumn("ner_pii_entities",    extract_ner_pii_entities(F.col("ner_chunk")))
    .withColumn("ner_filtered_keywords", filter_yake_by_ner(
        F.col("unique_keywords"), F.col("ner_pii_entities")
    ))
)

# ── Inspect: which YAKE keywords survived NER filtering? ─────────────────────
result_df.select(
    "text", "unique_keywords", "ner_pii_entities", "ner_filtered_keywords"
).show(truncate=60)

+------------------------------------------------------------+------------------------------------------------------------+---------------------------------+------------------------------------------------------------+
|                                                        text|                                             unique_keywords|                 ner_pii_entities|                                       ner_filtered_keywords|
+------------------------------------------------------------+------------------------------------------------------------+---------------------------------+------------------------------------------------------------+
|Quick, listened well, kind, helped me feel better, took c...|[quick, listened, well, kind, helped, feel, better, took,...|                               []|                                                          []|
|The nurse who did the triage was very rude to my friend. ...|[nurse, triage, rude, friend, already, stressed, embulato...|[

## 9. Apply Redaction

We compose two redaction steps:
1. **Regex pre-pass** — replaces structured PII tokens (dates, SSNs, IDs, times, postal codes)
2. **NER-filtered YAKE redaction** — replaces only those YAKE keywords that passed the NER filter
   (`PER`, `LOC`, `ORG`, `MISC`) with `[REDACTED]`

Non-PII YAKE keywords (e.g. `chest pain`, `complaint`, `clause`) are left intact.

In [12]:
def yake_redact(text, keywords):
    """
    Replace each keyword in *keywords* with [REDACTED] inside *text*.
    Uses whole-word, case-insensitive regex.
    Longer phrases are matched before their component words.
    """
    if not text or not keywords:
        return text
    for kw in sorted(keywords, key=len, reverse=True):
        text = re.sub(
            r'(\b%s\b)' % re.escape(kw),
            '[REDACTED]',
            text,
            flags=re.IGNORECASE,
        )
    return text

yake_redact_udf = F.udf(yake_redact, StringType())

# ── Compose: regex pre-pass → NER-filtered YAKE redaction ────────────────────
redacted_df = (
    result_df
    # Step 1: redact structured PII with regex
    .withColumn("step1_regex", regex_redact_udf(F.col("text")))
    # Step 2: redact only NER-filtered YAKE keywords
    .withColumn(
        "redacted_text",
        yake_redact_udf(F.col("step1_regex"), F.col("ner_filtered_keywords"))
    )
    .select(
        "text",
        "ner_filtered_keywords",
        "step1_regex",
        "redacted_text",
    )
)

redacted_df.select("text", "redacted_text").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|text                                                                                                                                                                                                                                             

## 10. Side-by-Side: Original → Regex-only → NER-filtered YAKE+Regex

In [13]:
rows = redacted_df.collect()
for r in rows:
    print("ORIGINAL       :", r["text"])
    print("REGEX ONLY     :", r["step1_regex"])
    print("FULL           :", r["redacted_text"])
    print("REDACTED KWS   :", r["ner_filtered_keywords"])
    print("-" * 110)

ORIGINAL       : Quick, listened well, kind, helped me feel better, took care of me quickly
REGEX ONLY     : Quick, listened well, kind, helped me feel better, took care of me quickly
FULL           : Quick, listened well, kind, helped me feel better, took care of me quickly
REDACTED KWS   : []
--------------------------------------------------------------------------------------------------------------
ORIGINAL       : The nurse who did the triage was very rude to my friend. Already she is stressed for being in an embulatory environment and in an other country (she is from Latino America). She doesn't understand French or English and she felt stress for coming here from the start. When the nurse took her vitals she hurt her arm. Now my friend has marks on her arm and she felt really bad after.
REGEX ONLY     : The nurse who did the triage was very rude to my friend. Already she is stressed for being in an embulatory environment and in an other country (she is from Latino America). She

## 11. HTML Visualization

Three-colour inline highlight — shows which spans are redacted and why:

| Colour | Meaning |
|---|---|
| 🟠 Orange `#FFB06E` | Regex-detected structured PII (dates, SSN, IDs, times, postal codes) |
| 🟢 Green `#90EE90` | YAKE keyword **and** NER `PER`/`LOC`/`ORG`/`MISC` → **will be redacted** |
| 🔵 Light-blue `#AED6F1` | NER `PER`/`LOC`/`ORG`/`MISC` entity **not** matched by any YAKE keyword |

Spans in grey (uncoloured) are YAKE keywords that did **not** pass the NER filter and are therefore left in the output.

In [14]:
from IPython.display import display, HTML

# ── LightPipeline for fast single-row annotation ─────────────────────────────
light_model = LightPipeline(combined_model)

# ── Compile regex patterns once for the visualiser ───────────────────────────
_COMPILED_REGEX = [
    (re.compile(pattern, re.IGNORECASE), label)
    for pattern, label in REGEX_PII_PATTERNS
]

_NER_PII_LABELS = {"PER", "LOC", "ORG", "MISC"}


def _build_html(text):
    """
    Returns an HTML string with three-layer inline highlighting:
      - Orange      : spans matched by the regex PII patterns
      - Green       : YAKE keywords that also appear as NER PER/LOC/ORG/MISC entities
      - Light-blue  : NER PER/LOC/ORG/MISC entities NOT matched by any YAKE keyword
    """
    annotations = light_model.fullAnnotate(text)[0]

    # ── Collect regex spans ───────────────────────────────────────────────────
    regex_spans = []  # (start, end, label)
    for compiled, label in _COMPILED_REGEX:
        for m in compiled.finditer(text):
            if not any(s <= m.start() and m.end() <= e for s, e, _ in regex_spans):
                regex_spans.append((m.start(), m.end(), label))

    # ── Collect NER PII entities ──────────────────────────────────────────────
    ner_pii = [
        chunk for chunk in annotations["ner_chunk"]
        if chunk.metadata.get("entity", "") in _NER_PII_LABELS
    ]
    ner_pii_texts_lower = {chunk.result.lower() for chunk in ner_pii}

    # ── Collect YAKE keyword spans & filter by NER overlap ───────────────────
    seen_kws = set()
    yake_ner_spans  = []  # green  : YAKE ∩ NER-PII
    keywords_sorted = sorted(
        annotations["keywords"], key=lambda k: len(k.result), reverse=True
    )
    for kw in keywords_sorted:
        kw_text  = kw.result
        kw_lower = kw_text.lower()
        if kw_lower in seen_kws:
            continue
        seen_kws.add(kw_lower)
        # Check NER overlap
        is_pii = any(
            kw_lower == ne or kw_lower in ne or ne in kw_lower
            for ne in ner_pii_texts_lower
        )
        if not is_pii:
            continue
        for m in re.finditer(r'(\b%s\b)' % re.escape(kw_text), text, re.IGNORECASE):
            if not any(s <= m.start() and m.end() <= e for s, e, _ in regex_spans):
                yake_ner_spans.append((m.start(), m.end(), kw_text))

    # ── NER-only spans (not covered by YAKE or regex) ─────────────────────────
    claimed = (
        [(s, e) for s, e, _ in regex_spans] +
        [(s, e) for s, e, _ in yake_ner_spans]
    )
    ner_only_spans = []
    for chunk in ner_pii:
        for m in re.finditer(
            r'(\b%s\b)' % re.escape(chunk.result), text, re.IGNORECASE
        ):
            if not any(s <= m.start() and m.end() <= e for s, e in claimed):
                ner_only_spans.append((m.start(), m.end(), chunk.metadata.get("entity", "")))

    # ── Merge all spans, resolve overlaps ─────────────────────────────────────
    all_spans = (
        [(s, e, lbl, "regex")    for s, e, lbl in regex_spans] +
        [(s, e, kw,  "yake_ner") for s, e, kw  in yake_ner_spans] +
        [(s, e, lbl, "ner_only") for s, e, lbl in ner_only_spans]
    )
    all_spans.sort(key=lambda x: x[0])
    merged, last_end = [], 0
    for span in all_spans:
        if span[0] >= last_end:
            merged.append(span)
            last_end = span[1]

    # ── Build HTML ────────────────────────────────────────────────────────────
    STYLE = {
        "regex":    ("background-color:#FFB06E; border-radius:3px; padding:1px 3px;"
                     " font-weight:bold; font-size:0.9em;"),
        "yake_ner": ("background-color:#90EE90; border-radius:3px; padding:1px 3px;"
                     " font-size:0.9em;"),
        "ner_only": ("background-color:#AED6F1; border-radius:3px; padding:1px 3px;"
                     " font-size:0.9em;"),
    }
    LABEL_STYLE = "font-size:0.7em; color:#555; vertical-align:super; margin-left:2px;"

    parts, cursor = [], 0
    for start, end, label, kind in merged:
        parts.append(text[cursor:start])
        parts.append(
            f'<span style="{STYLE[kind]}">'
            f'{text[start:end]}'
            f'<span style="{LABEL_STYLE}">[{label.upper()}]</span>'
            f'</span>'
        )
        cursor = end
    parts.append(text[cursor:])

    body = "".join(parts)
    return (
        '<div style="font-family:monospace; font-size:1em; line-height:1.8;'
        ' border:1px solid #ddd; border-radius:6px; padding:10px 14px;'
        f' margin-bottom:10px; background:#fafafa;">{body}</div>'
    )


# ── Legend ────────────────────────────────────────────────────────────────────
legend_html = (
    '<div style="font-family:sans-serif; font-size:0.85em; margin-bottom:12px;">'
    '<b>Legend:</b>&nbsp;'
    '<span style="background:#FFB06E; border-radius:3px; padding:1px 6px;">Regex PII → redacted</span>'
    '&nbsp;&nbsp;'
    '<span style="background:#90EE90; border-radius:3px; padding:1px 6px;">YAKE ∩ NER PER/LOC/ORG/MISC → redacted</span>'
    '&nbsp;&nbsp;'
    '<span style="background:#AED6F1; border-radius:3px; padding:1px 6px;">NER PER/LOC/ORG/MISC (no YAKE match)</span>'
    '</div>'
)
display(HTML(legend_html))

# ── Render each sample sentence ───────────────────────────────────────────────
for text in sample_texts:
    display(HTML(_build_html(text)))

## 12. Threshold Sensitivity Analysis

YAKE scores are **lower = more relevant**. The `setThreshold` parameter filters out keywords whose score
is above the threshold. The table below shows how coverage changes with different thresholds **after**
the NER filter is applied — only PER/LOC/ORG/MISC keywords are counted.

In [15]:
thresholds = [0.5, 1.0, 1.5, 2.5]
sample     = sample_texts[1]  # SSN + names text

print(f"Text: {sample}\n")
annotations = light_model.fullAnnotate(sample)[0]
ner_pii_lower = {
    chunk.result.lower()
    for chunk in annotations["ner_chunk"]
    if chunk.metadata.get("entity", "") in _NER_PII_LABELS
}
print(f"  NER PER/LOC/ORG/MISC entities : {sorted(ner_pii_lower)}\n")

for t in thresholds:
    # Filter YAKE keywords by threshold
    all_kws = [
        k.result
        for k in annotations["keywords"]
        if float(k.metadata["score"]) <= t
    ]
    all_kws = list(dict.fromkeys(all_kws))

    # Further filter by NER overlap
    ner_kws = [
        kw for kw in all_kws
        if any(
            kw.lower() == ne or kw.lower() in ne or ne in kw.lower()
            for ne in ner_pii_lower
        )
    ]
    print(
        f"  threshold={t:<4}  "
        f"YAKE total={len(all_kws):>2}  "
        f"after NER filter={len(ner_kws):>2}  "
        f"keywords: {ner_kws}"
    )

Text: The nurse who did the triage was very rude to my friend. Already she is stressed for being in an embulatory environment and in an other country (she is from Latino America). She doesn't understand French or English and she felt stress for coming here from the start. When the nurse took her vitals she hurt her arm. Now my friend has marks on her arm and she felt really bad after.

  NER PER/LOC/ORG/MISC entities : ['english', 'french', 'latino america']

  threshold=0.5   YAKE total= 5  after NER filter= 1  keywords: ['latino america']
  threshold=1.0   YAKE total=27  after NER filter= 6  keywords: ['latino', 'america', 'french', 'english', 'latino america', 'understand french']
  threshold=1.5   YAKE total=30  after NER filter= 7  keywords: ['latino', 'america', 'french', 'english', 'latino america', 'understand french', 'french or english']
  threshold=2.5   YAKE total=30  after NER filter= 7  keywords: ['latino', 'america', 'french', 'english', 'latino america', 'understand fre

## 13. (Optional) Save Redacted Output to S3

Uses the **SageMaker v2.x** `sagemaker.s3.S3Uploader` API.

In [ ]:
import tempfile
from sagemaker.s3 import S3Uploader

local_output = os.path.join(tempfile.mkdtemp(), "redacted_yake_ner_output.csv")
redacted_df.select("text", "redacted_text").toPandas().to_csv(local_output, index=False)

s3_uri = f"s3://{bucket}/{prefix}/redacted_yake_ner_output.csv"

S3Uploader.upload(
    local_path=local_output,
    desired_s3_uri=s3_uri,
    sagemaker_session=sm_session,
)
print(f"Redacted output uploaded to: {s3_uri}")

## 14. Cleanup

In [16]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
